# Dataset 3: Multivariate Gait Data Classification

This notebook covers time-series classification on biomechanical gait data. Ten subjects walked under three conditions (Unbraced, Knee Braced, Ankle Braced), and the task is to classify the walking condition from joint angle patterns.

Key challenges: multivariate time-series, feature engineering vs deep learning, subject generalization.

## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_val_score, LeaveOneGroupOut)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score

import warnings
warnings.filterwarnings('ignore')
print('All libraries loaded.')

## Step 2 — Load and Explore the Dataset

In [ ]:
df = pd.read_csv('gait_data.csv')
print(f'Dataset Shape: {df.shape}')
print(f'\nColumns: {list(df.columns)}')
print(f'\nCondition Distribution:')
print(df['Condition'].value_counts())
print(f'\nSubjects: {df["Subject_ID"].nunique()}')
print(f'Gait Cycles per subject per condition: {df["Gait_Cycle"].nunique()}')
df.head()

In [ ]:
# Basic statistics
print('Joint Angle Statistics:')
angle_cols = ['Ankle_Left','Ankle_Right','Knee_Left','Knee_Right','Hip_Left','Hip_Right']
print(df[angle_cols].describe().round(2))

## Step 3 — Visualize Gait Patterns

In [ ]:
# Average gait cycle per condition
fig, axes = plt.subplots(3, 2, figsize=(16, 12))
joint_pairs = [('Ankle_Left','Ankle_Right'), ('Knee_Left','Knee_Right'), ('Hip_Left','Hip_Right')]
joint_names = ['Ankle', 'Knee', 'Hip']
colors = {'Unbraced': '#2ecc71', 'Knee_Braced': '#e74c3c', 'Ankle_Braced': '#3498db'}

for row, (left_col, right_col) in enumerate(joint_pairs):
    for col_idx, (side_col, side_name) in enumerate([(left_col, 'Left'), (right_col, 'Right')]):
        ax = axes[row][col_idx]
        for cond in ['Unbraced', 'Knee_Braced', 'Ankle_Braced']:
            subset = df[df['Condition'] == cond]
            avg = subset.groupby('Time_Percent')[side_col].mean()
            ax.plot(avg.index, avg.values, label=cond, color=colors[cond], linewidth=2)
        ax.set_title(f'{joint_names[row]} — {side_name}', fontsize=12)
        ax.set_xlabel('Gait Cycle (%)')
        ax.set_ylabel('Angle (degrees)')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

plt.suptitle('Average Joint Angles Across Walking Conditions', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Subject variability — plot individual gait cycles for one condition
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for idx, subj in enumerate(range(1, 11)):
    ax = axes[idx // 5][idx % 5]
    subset = df[(df['Subject_ID'] == subj) & (df['Condition'] == 'Unbraced')]
    for cycle in range(1, 11):
        cycle_data = subset[subset['Gait_Cycle'] == cycle]
        ax.plot(cycle_data['Time_Percent'], cycle_data['Knee_Left'], alpha=0.5, linewidth=0.8)
    ax.set_title(f'Subject {subj}', fontsize=10)
    ax.set_xlabel('Gait %')
    ax.set_ylabel('Knee L (deg)')

plt.suptitle('Individual Gait Cycles — Knee Left (Unbraced)', fontsize=14)
plt.tight_layout()
plt.show()

## Step 4 — Feature Engineering

Extract statistical features from each gait cycle to convert time-series data into a tabular format suitable for traditional ML classifiers.

In [ ]:
# Extract features per gait cycle
def extract_features(group):
    features = {}
    for col in angle_cols:
        values = group[col].values
        features[f'{col}_mean'] = np.mean(values)
        features[f'{col}_std'] = np.std(values)
        features[f'{col}_min'] = np.min(values)
        features[f'{col}_max'] = np.max(values)
        features[f'{col}_range'] = np.max(values) - np.min(values)
        features[f'{col}_median'] = np.median(values)
        features[f'{col}_skew'] = pd.Series(values).skew()
        features[f'{col}_kurtosis'] = pd.Series(values).kurtosis()
    return pd.Series(features)

# Group by subject, condition, and gait cycle
feature_df = df.groupby(['Subject_ID', 'Condition', 'Gait_Cycle']).apply(extract_features).reset_index()

print(f'Feature Matrix Shape: {feature_df.shape}')
print(f'Features per cycle: {feature_df.shape[1] - 3}')
print(f'Total gait cycles: {len(feature_df)}')
feature_df.head()

## Step 5 — Prepare Data for Classification

In [ ]:
# Encode target
le = LabelEncoder()
feature_df['Condition_Encoded'] = le.fit_transform(feature_df['Condition'])
print('Classes:', dict(zip(le.classes_, le.transform(le.classes_))))

# Separate features and target
feature_cols = [c for c in feature_df.columns if c not in ['Subject_ID','Condition','Gait_Cycle','Condition_Encoded']]
X = feature_df[feature_cols].values
y = feature_df['Condition_Encoded'].values
groups = feature_df['Subject_ID'].values  # For leave-one-subject-out

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'\nX shape: {X_scaled.shape}')
print(f'y shape: {y.shape}')
print(f'\nClass distribution: {dict(zip(*np.unique(y, return_counts=True)))}')

In [ ]:
# Standard train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Training set: {X_train.shape}')
print(f'Test set: {X_test.shape}')

## Step 6 — Classification Models

In [ ]:
# Model 1: Random Forest
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print('MODEL 1 — Random Forest')
print('=' * 40)
print(classification_report(y_test, y_pred_rf, target_names=le.classes_))
print(f'Macro F1: {f1_score(y_test, y_pred_rf, average="macro"):.4f}')

In [ ]:
# Model 2: SVM with RBF kernel
svm = SVC(kernel='rbf', random_state=42)
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)

print('MODEL 2 — SVM (RBF Kernel)')
print('=' * 40)
print(classification_report(y_test, y_pred_svm, target_names=le.classes_))
print(f'Macro F1: {f1_score(y_test, y_pred_svm, average="macro"):.4f}')

In [ ]:
# Model 3: Gradient Boosting
gb = GradientBoostingClassifier(n_estimators=200, random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)

print('MODEL 3 — Gradient Boosting')
print('=' * 40)
print(classification_report(y_test, y_pred_gb, target_names=le.classes_))
print(f'Macro F1: {f1_score(y_test, y_pred_gb, average="macro"):.4f}')

In [ ]:
# Model 4: KNN
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)

print('MODEL 4 — K-Nearest Neighbors (k=5)')
print('=' * 45)
print(classification_report(y_test, y_pred_knn, target_names=le.classes_))
print(f'Macro F1: {f1_score(y_test, y_pred_knn, average="macro"):.4f}')

## Step 7 — Model Comparison

In [ ]:
# Summary comparison
all_results = {
    'Random Forest': f1_score(y_test, y_pred_rf, average='macro'),
    'SVM (RBF)': f1_score(y_test, y_pred_svm, average='macro'),
    'Gradient Boosting': f1_score(y_test, y_pred_gb, average='macro'),
    'KNN': f1_score(y_test, y_pred_knn, average='macro')
}

plt.figure(figsize=(10, 5))
colors = ['#3498db','#e74c3c','#2ecc71','#f39c12']
plt.bar(all_results.keys(), all_results.values(), color=colors)
plt.ylabel('Macro F1-Score')
plt.title('Model Comparison — Gait Classification', fontsize=14)
plt.ylim(0, 1)
for i, (name, score) in enumerate(all_results.items()):
    plt.text(i, score + 0.02, f'{score:.3f}', ha='center', fontsize=12)
plt.tight_layout()
plt.show()

## Step 8 — Feature Importance Analysis

In [ ]:
# Feature importance from Random Forest
importances = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(12, 10))
plt.barh(importances['Feature'][:20], importances['Importance'][:20], color='teal')
plt.xlabel('Importance')
plt.title('Top 20 Feature Importances — Random Forest', fontsize=14)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print('Top 10 Features:')
print(importances.head(10).to_string(index=False))

## Step 9 — Leave-One-Subject-Out Cross-Validation

This is the real test of generalization. Standard cross-validation mixes gait cycles from the same subject across train and test, which leaks subject-specific patterns. Leave-one-subject-out tests whether the model works on completely unseen individuals.

In [ ]:
# Leave-One-Subject-Out CV
logo = LeaveOneGroupOut()

logo_results = {}
for name, model in [('Random Forest', RandomForestClassifier(n_estimators=200, random_state=42)),
                     ('SVM', SVC(kernel='rbf', random_state=42)),
                     ('Gradient Boosting', GradientBoostingClassifier(n_estimators=100, random_state=42))]:
    scores = cross_val_score(model, X_scaled, y, cv=logo, groups=groups, scoring='f1_macro')
    logo_results[name] = scores
    print(f'{name}:')
    print(f'  LOSO CV Macro F1: {scores.mean():.4f} (+/- {scores.std():.4f})')
    print(f'  Per-subject scores: {[round(s, 3) for s in scores]}')
    print()

In [ ]:
# Compare standard CV vs LOSO CV
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Standard 5-fold CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
std_scores = {}
for name, model in [('RF', RandomForestClassifier(n_estimators=200, random_state=42)),
                     ('SVM', SVC(kernel='rbf', random_state=42))]:
    std_scores[name] = cross_val_score(model, X_scaled, y, cv=cv, scoring='f1_macro')

axes[0].boxplot(std_scores.values(), labels=std_scores.keys())
axes[0].set_title('Standard 5-Fold CV', fontsize=13)
axes[0].set_ylabel('Macro F1-Score')

axes[1].boxplot([logo_results['Random Forest'], logo_results['SVM']], labels=['RF','SVM'])
axes[1].set_title('Leave-One-Subject-Out CV', fontsize=13)
axes[1].set_ylabel('Macro F1-Score')

plt.suptitle('Standard CV vs LOSO CV Comparison', fontsize=15)
plt.tight_layout()
plt.show()

## Step 10 — Confusion Matrix (Best Model)

In [ ]:
# Best model confusion matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm, annot=True, fmt='d', cmap='YlGnBu',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix — Random Forest', fontsize=14)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

# Normalized version
plt.figure(figsize=(8, 6))
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='YlGnBu',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix — Normalized', fontsize=14)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

## Summary and Key Takeaways

### Results Summary

- Statistical feature extraction converts time-series into a tabular classification problem
- Random Forest and Gradient Boosting handle the extracted features well
- Standard CV scores are optimistic because they mix same-subject data
- Leave-one-subject-out CV gives the realistic generalization estimate

### What This Dataset Taught You

1. Time-series classification starts with smart feature engineering
2. Mean, std, range, skew, and kurtosis capture most of the gait cycle information
3. Leave-one-subject-out is the correct evaluation for biomedical data with few subjects
4. The gap between standard CV and LOSO CV reveals how much subject-specific information your model uses
5. Feature importance shows which joints and which statistical properties drive classification